# RS-Flow-VQA: Continuous Soft-Prefix Flow Matching & FreeFlow Distillation

This notebook demonstrates the complete end-to-end pipeline for **RS-Flow-VQA**:
1. **Data Preparation**: Loading & tokenizing RSICD dataset captions.
2. **Feature & Embedding Caching**: Extracting Scale-MAE 1024-dim image features and Qwen token lookup table.
3. **Whitening Normalization**: Computing per-channel mean and standard deviation for prompt embedding whitening.
4. **Conditional Flow Matching (CFM) Teacher**: Training CFM teacher $v_\phi$ with Minibatch OT coupling.
5. **Target-Free FreeFlow Distillation**: Distilling into 1-step student $f_\theta$ using discrete prediction ($N=8$) and auxiliary correction $g_\psi$.
6. **Evaluation & Zero-Shot VQA Transfer**: Metric calculation on RSICD captions and zero-shot VQA on RSVQA-LR.

In [ ]:
# Install uv when needed, then install into the active Colab/Jupyter kernel.
import shutil
import subprocess

if shutil.which('uv') is None:
    subprocess.run('curl -LsSf https://astral.sh/uv/install.sh | sh', shell=True, check=True)
uv_bin = shutil.which('uv') or '/root/.local/bin/uv'
subprocess.run([uv_bin, 'pip', 'install', '--system', '-e', '.[gpu,notebook]'], check=True)

In [ ]:
import torch
from rs_flow_vqa.config import load_config
from rs_flow_vqa.training.train_teacher import train_teacher_pipeline
from rs_flow_vqa.training.distill_freeflow import distill_freeflow_pipeline
from rs_flow_vqa.evaluation.eval_caption import evaluate_caption_pipeline
from rs_flow_vqa.evaluation.eval_rsvqa import evaluate_rsvqa_pipeline

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())

## 1. Choose execution profile
Set `RUN_SMOKE=False` for the real T4 pipeline after placing RSICD and RSVQA-LR under the paths configured in `configs/t4.yaml`.

In [ ]:
RUN_SMOKE = True
cfg = load_config(
    'configs/smoke.yaml' if RUN_SMOKE else 'configs/t4.yaml',
    smoke=RUN_SMOKE,
    device_override='cpu' if RUN_SMOKE else 'cuda',
)
print('Loaded Experiment:', cfg.experiment_name)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Fail before downloading models or training if smoke fixtures were
# accidentally selected for a real experiment.
if not RUN_SMOKE:
    from rs_flow_vqa.data.rsicd import RSICDDataset
    from rs_flow_vqa.data.rsvqa import RSVQADataset
    rsicd_check = RSICDDataset(cfg.data.rsicd_data_dir, split='all')
    rsvqa_check = RSVQADataset(cfg.data.rsvqa_data_dir, split='test')
    print(f'Real-data check: {len(rsicd_check)} RSICD captions, {len(rsvqa_check)} RSVQA test questions')

In [ ]:
# Step 1: Feature caching
from rs_flow_vqa.cli import cache_features_cmd
import argparse

args = argparse.Namespace(
    config='configs/smoke.yaml' if RUN_SMOKE else 'configs/t4.yaml',
    smoke=RUN_SMOKE,
    device='cpu' if RUN_SMOKE else 'cuda',
    seed=42,
    output_dir=cfg.output_dir,
)
cache_features_cmd(args)

In [ ]:
# Step 2: Train CFM Teacher
teacher_ckpt = train_teacher_pipeline(cfg)

In [ ]:
# Step 3: Target-Free FreeFlow Distillation
student_ckpt = distill_freeflow_pipeline(cfg)

In [ ]:
# Step 4: Evaluate Caption Quality & Fidelity
cap_metrics = evaluate_caption_pipeline(cfg)

In [ ]:
# Step 5: Evaluate Zero-Shot VQA Transfer on RSVQA-LR
rsvqa_metrics = evaluate_rsvqa_pipeline(cfg)

## 2. Summary Metrics Table
Print summary metrics table comparing Baselines, 16-NFE Teacher, and 1-Step FreeFlow Student.

In [ ]:
import pandas as pd

data = [
    {'Model': 'Text-Only Qwen Baseline', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['text_only_baseline']['overall']*100:.2f}%", 'Bridge Latency (ms)': '0.0 ms', 'NFEs': 0},
    {'Model': 'CFM Teacher (16-NFE)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['teacher_16nfe']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['teacher_16nfe_latency_ms']:.2f} ms", 'NFEs': 16},
    {'Model': 'FreeFlow Student (1-Step)', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['student_1step']['overall']*100:.2f}%", 'Bridge Latency (ms)': f"{cap_metrics['student_1step_latency_ms']:.2f} ms", 'NFEs': 1},
    {'Model': 'Wrong-Image Control', 'RSVQA Accuracy (%)': f"{rsvqa_metrics['shuffled_image_teacher_control']['overall']*100:.2f}%", 'Bridge Latency (ms)': '-', 'NFEs': 16},
]
df = pd.DataFrame(data)
display(df)
print('Teacher endpoint condition gap:', f"{cap_metrics['endpoint_condition_gap']:.2%}")
print('Prefix-length MAE:', f"{cap_metrics['prefix_length_mae']:.2f} tokens")